In [ ]:
!pip install pyspark

In [ ]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time, os

uploaded = files.upload()
spark = SparkSession.builder.appName("EnergyAssessmentSet03").getOrCreate()

def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)

readings_df = load_csv("meter_readings_90000.csv")
readings_extra_df = load_csv("meter_readings_extra_12000.csv")
meter_df = load_csv("meter_master.csv")
substation_df = load_csv("substation_master.csv")
tariff_df = load_csv("tariff_master.csv")
maintenance_df = load_csv("maintenance_tickets_1600.csv")


Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv
Saving meter_master.csv to meter_master.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving substation_master.csv to substation_master.csv
Saving tariff_master.csv to tariff_master.csv


Section 01 — Data Preparation & Analysis

A. Inspect, select and transform

In [ ]:
readings_df.printSchema()
print("Main readings row count:", readings_df.count())
readings_df.show(10)


root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Main readings row count: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|  

In [ ]:
selected_readings_df = readings_df.select(
    "reading_id", "meter_id", "reading_timestamp", "energy_kwh", "reading_status"
)
renamed_source_df = readings_df.withColumnRenamed("source_type", "collection_mode")
selected_readings_df.show(5)
renamed_source_df.select("reading_id", "collection_mode").show(5)


+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|         Valid|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|         Valid|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|     Estimated|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|         Valid|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|         Valid|
+----------+--------+-------------------+----------+--------------+
only showing top 5 rows
+----------+---------------+
|reading_id|collection_mode|
+----------+---------------+
| RD0000001|     SmartMeter|
| RD0000002|     SmartMeter|
| RD0000003|     SmartMeter|
| RD0000004|         Manual|
| RD0000005|     SmartMeter|
+----------+---------------+
only showing top 5 rows


In [ ]:
valid_high_energy_df = readings_df.filter(
    (F.col("reading_status") == "Valid") & (F.col("energy_kwh") > 20)
).orderBy(F.col("energy_kwh").desc())
valid_high_energy_df.show(20)


+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid| SmartMeter|                 31.1|      OP010|
| RD0077494|   M0014|2026-05-05 17:30:00|    109.99|     NULL|       0.771|             0|         Valid|     Manual|                 39.0|      OP031|
| RD0034412|   M0026|2026-02-24 21:15:00|    109.96|    231.6|       0.964|             0|         Valid| SmartMeter|                 25.3|      OP017|
| RD0039063|   M0101|2026-02-01 00:45:00|    109.96|    231.6|       0.835|             

In [ ]:
energy_band_df = readings_df.withColumn(
    "energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium")
     .otherwise("High")
)
energy_band_df.select("reading_id", "energy_kwh", "energy_band").show(10)


+----------+----------+-----------+
|reading_id|energy_kwh|energy_band|
+----------+----------+-----------+
| RD0000001|     22.53|     Medium|
| RD0000002|      2.97|        Low|
| RD0000003|      8.61|     Medium|
| RD0000004|      1.87|        Low|
| RD0000005|      4.39|        Low|
| RD0000006|      0.36|        Low|
| RD0000007|     79.47|       High|
| RD0000008|     28.35|       High|
| RD0000009|      1.96|        Low|
| RD0000010|     22.52|     Medium|
+----------+----------+-----------+
only showing top 10 rows


B. Null values and data cleaning

In [ ]:
null_counts_df = readings_df.select(*[
    F.sum(F.col(c).isNull().cast("long")).alias(c + "_null_count")
    for c in ["energy_kwh", "voltage_v", "power_factor", "operator_id"]
])
null_counts_df.show()


+---------------------+--------------------+-----------------------+----------------------+
|energy_kwh_null_count|voltage_v_null_count|power_factor_null_count|operator_id_null_count|
+---------------------+--------------------+-----------------------+----------------------+
|                 1620|                2070|                   1440|                  5400|
+---------------------+--------------------+-----------------------+----------------------+



In [ ]:
clean_defaults_df = readings_df.fillna({
    "operator_id": "UNASSIGNED",
    "power_factor": 0.90
})
clean_defaults_df.select("operator_id", "power_factor").show(10)


+-----------+------------+
|operator_id|power_factor|
+-----------+------------+
|      OP029|       0.939|
|      OP025|       0.845|
|      OP007|       0.879|
| UNASSIGNED|       0.831|
|      OP019|       0.966|
|      OP021|       0.788|
|      OP016|       0.986|
|      OP025|       0.818|
|      OP010|       0.817|
|      OP028|       0.804|
+-----------+------------+
only showing top 10 rows


In [ ]:
before_clean_count = readings_df.count()
readings_nonnull_df = readings_df.filter(
    F.col("energy_kwh").isNotNull() & F.col("voltage_v").isNotNull()
)
after_clean_count = readings_nonnull_df.count()
print("Rows before:", before_clean_count)
print("Rows after:", after_clean_count)
print("Rows removed:", before_clean_count - after_clean_count)


Rows before: 90000
Rows after: 86349
Rows removed: 3651


C. Joins, groupBy and aggregations

In [ ]:
readings_meter_df = readings_df.join(meter_df, on="meter_id", how="left")
valid_energy_by_site_df = (
    readings_meter_df.filter(F.col("reading_status") == "Valid")
    .groupBy("site_type")
    .agg(F.sum("energy_kwh").alias("total_valid_energy_kwh"))

)
valid_energy_by_site_df.show()


+---------------+----------------------+
|      site_type|total_valid_energy_kwh|
+---------------+----------------------+
|Public Facility|             138870.41|
|    Residential|     100863.2900000004|
|     Industrial|     710317.4100000021|
|     Commercial|     451525.1799999996|
+---------------+----------------------+



In [ ]:
readings_meter_substation_df = readings_meter_df.join(
    substation_df, on="substation_id", how="left"
)
valid_energy_region_city_df = (
    readings_meter_substation_df.filter(F.col("reading_status") == "Valid")
    .groupBy("region", "city")
    .agg(F.sum("energy_kwh").alias("total_valid_energy_kwh"))

)
valid_energy_region_city_df.show()


+-------+-------------+----------------------+
| region|         city|total_valid_energy_kwh|
+-------+-------------+----------------------+
|Central|       Indore|    176503.91000000012|
|   West|       Mumbai|     52116.20999999999|
|  South|    Hyderabad|    51937.789999999884|
|  South|Visakhapatnam|    205015.82999999932|
|   West|    Ahmedabad|     138627.7400000004|
|  North|      Lucknow|     142610.3999999998|
|  South|    Bengaluru|     73787.59999999998|
|  North|       Jaipur|     47863.15999999996|
|  South|        Kochi|     204272.4600000002|
|  North|        Delhi|     94234.89000000014|
|  South|      Chennai|     51829.05999999996|
|   West|         Pune|    162777.23999999996|
+-------+-------------+----------------------+



In [ ]:
readings_tariff_df = readings_meter_df.join(tariff_df, on="tariff_id", how="left")
billable_readings_df = (
    readings_tariff_df
    .filter((F.col("reading_status") != "Rejected") & F.col("energy_kwh").isNotNull())
    .withColumn("energy_charge_inr", F.col("energy_kwh") * F.col("rate_per_kwh_inr"))
)
billable_readings_df.select(
    "reading_id", "meter_id", "reading_status", "energy_kwh",
    "rate_per_kwh_inr", "energy_charge_inr"
).show(10)


+----------+--------+--------------+----------+----------------+------------------+
|reading_id|meter_id|reading_status|energy_kwh|rate_per_kwh_inr| energy_charge_inr|
+----------+--------+--------------+----------+----------------+------------------+
| RD0000001|   M0082|         Valid|     22.53|             6.7|150.95100000000002|
| RD0000002|   M0020|         Valid|      2.97|             6.4|19.008000000000003|
| RD0000003|   M0054|     Estimated|      8.61|             8.2| 70.60199999999999|
| RD0000004|   M0111|         Valid|      1.87|             5.1| 9.536999999999999|
| RD0000005|   M0135|         Valid|      4.39|             8.2|            35.998|
| RD0000006|   M0139|         Valid|      0.36|             6.4|             2.304|
| RD0000007|   M0142|         Valid|     79.47|             9.1| 723.1769999999999|
| RD0000009|   M0005|     Estimated|      1.96|             5.1| 9.995999999999999|
| RD0000010|   M0052|         Valid|     22.52|             6.7|150.88400000

In [ ]:
tickets_meter_df = maintenance_df.join(meter_df, on="meter_id", how="left")
ticket_summary_df = (
    tickets_meter_df.groupBy("site_type", "severity")
    .agg(
        F.count("ticket_id").alias("ticket_count"),
        F.avg("maintenance_cost_inr").alias("avg_maintenance_cost_inr")
    )

)
ticket_summary_df.show()


+---------------+--------+------------+------------------------+
|      site_type|severity|ticket_count|avg_maintenance_cost_inr|
+---------------+--------+------------+------------------------+
|    Residential|Critical|          41|      11875.782702702705|
|     Industrial|  Medium|         106|      12043.898526315788|
|     Commercial|Critical|          37|      13424.726216216219|
|     Commercial|     Low|         166|      12223.517692307689|
|Public Facility|    High|          48|      12905.279347826086|
|    Residential|     Low|         194|       12531.90201117318|
|Public Facility|  Medium|          97|      13050.628421052634|
|     Industrial|Critical|          17|      12296.198000000002|
|Public Facility|Critical|          15|       8986.843571428572|
|     Industrial|    High|          63|      12838.869344262299|
|    Residential|    High|         156|      11047.967122302161|
|Public Facility|     Low|          60|      11195.929464285715|
|     Commercial|  Medium

In [ ]:
substation_reliability_df = (
    readings_meter_df.groupBy("substation_id")
    .agg(
        F.avg("voltage_v").alias("average_voltage_v"),
        F.sum("outage_minutes").alias("total_outage_minutes")
    )
    .orderBy(F.col("total_outage_minutes").desc())
)
substation_reliability_df.show(5)


+-------------+------------------+--------------------+
|substation_id| average_voltage_v|total_outage_minutes|
+-------------+------------------+--------------------+
|        SUB10|230.04010567581344|               91224|
|        SUB09|230.04586272640606|               81734|
|        SUB04| 230.0558687466385|               75872|
|        SUB08|229.88012557077673|               75581|
|        SUB03|230.03094896227708|               61662|
+-------------+------------------+--------------------+
only showing top 5 rows


In [ ]:
status_counts_by_source_df = (
    readings_df.filter(F.col("reading_status").isin("Valid", "Estimated", "Rejected"))
    .groupBy("source_type")
    .pivot("reading_status", ["Valid", "Estimated", "Rejected"])
    .count()
    .na.fill(0)
    .withColumn("reading_count", F.col("Valid") + F.col("Estimated") + F.col("Rejected"))
    .orderBy(F.col("reading_count").desc())
)
status_counts_by_source_df.show(truncate=False)


+-----------+-----+---------+--------+-------------+
|source_type|Valid|Estimated|Rejected|reading_count|
+-----------+-----+---------+--------+-------------+
|SmartMeter |65521|10443    |3202    |79166        |
|Manual     |8965 |1405     |464     |10834        |
+-----------+-----+---------+--------+-------------+



Section 02 — Advanced DataFrame Operations

D. Date and window functions

In [ ]:
readings_ts_df = readings_meter_substation_df.withColumn(
    "reading_timestamp", F.to_timestamp("reading_timestamp")
).withColumn("reading_year", F.year("reading_timestamp")).withColumn("reading_month", F.month("reading_timestamp"))
monthly_valid_energy_region_df = (
    readings_ts_df.filter(F.col("reading_status") == "Valid")
    .groupBy("region", "reading_year", "reading_month")
    .agg(F.sum("energy_kwh").alias("monthly_valid_energy_kwh"))

)
monthly_valid_energy_region_df.show(20)


+-------+------------+-------------+------------------------+
| region|reading_year|reading_month|monthly_valid_energy_kwh|
+-------+------------+-------------+------------------------+
|Central|        2026|            5|      30048.879999999986|
|   West|        2026|            1|       59967.56999999997|
|  South|        2026|            1|       97547.38999999987|
|Central|        2026|            6|       27536.84999999999|
|  North|        2026|            3|       51094.61000000006|
|Central|        2026|            2|      28161.410000000007|
|  North|        2026|            2|      45444.249999999985|
|  South|        2026|            2|       90864.80000000006|
|   West|        2026|            4|       57591.23999999994|
|  South|        2026|            5|       101204.8699999999|
|Central|        2026|            3|                29805.85|
|   West|        2026|            5|       61006.62999999993|
|Central|        2026|            4|      29475.339999999982|
|   West

In [ ]:
meter_valid_totals_df = (
    readings_meter_df.filter(F.col("reading_status") == "Valid")
    .groupBy("substation_id", "meter_id")
    .agg(F.sum("energy_kwh").alias("total_valid_energy_kwh"))
)
rank_window = Window.partitionBy("substation_id").orderBy(F.col("total_valid_energy_kwh").desc())
top3_meters_df = meter_valid_totals_df.withColumn(
    "energy_rank", F.dense_rank().over(rank_window)
).filter(F.col("energy_rank") <= 3).orderBy("substation_id", "energy_rank")
top3_meters_df.show()


+-------------+--------+----------------------+-----------+
|substation_id|meter_id|total_valid_energy_kwh|energy_rank|
+-------------+--------+----------------------+-----------+
|SUB01        |M0130   |9364.98               |1          |
|SUB01        |M0110   |9259.750000000002     |2          |
|SUB01        |M0068   |8832.580000000005     |3          |
|SUB02        |M0071   |29242.04              |1          |
|SUB02        |M0145   |28641.319999999992    |2          |
|SUB02        |M0133   |7060.140000000003     |3          |
|SUB03        |M0096   |9632.740000000003     |1          |
|SUB03        |M0080   |9525.940000000002     |2          |
|SUB03        |M0073   |9369.789999999995     |3          |
|SUB04        |M0038   |29505.88              |1          |
|SUB04        |M0099   |29216.319999999996    |2          |
|SUB04        |M0098   |28613.460000000014    |3          |
|SUB05        |M0094   |9659.639999999996     |1          |
|SUB05        |M0050   |9570.85999999999

In [ ]:
valid_timestamp_df = readings_df.filter(F.col("reading_status") == "Valid").withColumn(
    "reading_timestamp", F.to_timestamp("reading_timestamp")
)
cumulative_window = (
    Window.partitionBy("meter_id")
    .orderBy(F.col("reading_timestamp"), F.col("reading_id"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
cumulative_energy_df = valid_timestamp_df.withColumn(
    "cumulative_valid_energy_kwh",
    F.sum("energy_kwh").over(cumulative_window)
)
cumulative_energy_df.select(
    "meter_id", "reading_timestamp", "reading_id", "energy_kwh", "cumulative_valid_energy_kwh"
).show(20, truncate=False)


+--------+-------------------+----------+----------+---------------------------+
|meter_id|reading_timestamp  |reading_id|energy_kwh|cumulative_valid_energy_kwh|
+--------+-------------------+----------+----------+---------------------------+
|M0001   |2026-01-01 01:15:00|RD0083587 |22.27     |22.27                      |
|M0001   |2026-01-01 03:30:00|RD0086275 |4.05      |26.32                      |
|M0001   |2026-01-01 09:45:00|RD0088204 |11.28     |37.6                       |
|M0001   |2026-01-02 04:45:00|RD0005149 |11.76     |49.36                      |
|M0001   |2026-01-02 08:15:00|RD0035530 |10.06     |59.42                      |
|M0001   |2026-01-02 18:30:00|RD0070104 |15.83     |75.25                      |
|M0001   |2026-01-03 00:30:00|RD0006677 |17.06     |92.31                      |
|M0001   |2026-01-03 06:00:00|RD0021042 |17.14     |109.45                     |
|M0001   |2026-01-03 08:30:00|RD0032205 |20.46     |129.91                     |
|M0001   |2026-01-03 18:15:0

E. Union and deduplication

In [25]:
main_count = readings_df.count()
extra_count = readings_extra_df.count()
combined_readings_df = readings_df.unionByName(readings_extra_df)
combined_count = combined_readings_df.count()
print("Main file rows:", main_count)
print("Extra file rows:", extra_count)
print("Expected sum:", main_count + extra_count)
print("After union:", combined_count)


Main file rows: 90000
Extra file rows: 12000
Expected sum: 102000
After union: 102000


In [26]:
repeated_reading_ids_df = (
    combined_readings_df.groupBy("reading_id")
    .count().filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)
print("Repeated reading_id values:")
repeated_reading_ids_df.show(50)
unique_readings_df = combined_readings_df.dropDuplicates(["reading_id"])
print("Rows after deduplication:", unique_readings_df.count())


Repeated reading_id values:
+----------+-----+
|reading_id|count|
+----------+-----+
|RD0025848 |2    |
|RD0045533 |2    |
|RD0064921 |2    |
|RD0056575 |2    |
|RD0058815 |2    |
|RD0022131 |2    |
|RD0029598 |2    |
|RD0045710 |2    |
|RD0030877 |2    |
|RD0002679 |2    |
|RD0011169 |2    |
|RD0028736 |2    |
|RD0030669 |2    |
|RD0061933 |2    |
|RD0015904 |2    |
|RD0008188 |2    |
|RD0037899 |2    |
|RD0000004 |2    |
|RD0025628 |2    |
|RD0048911 |2    |
|RD0055647 |2    |
|RD0000194 |2    |
|RD0004157 |2    |
|RD0020282 |2    |
|RD0040813 |2    |
|RD0056989 |2    |
|RD0062676 |2    |
|RD0015570 |2    |
|RD0067957 |2    |
|RD0014026 |2    |
|RD0033864 |2    |
|RD0043991 |2    |
|RD0066172 |2    |
|RD0021466 |2    |
|RD0050432 |2    |
|RD0026433 |2    |
|RD0032257 |2    |
|RD0058086 |2    |
|RD0009162 |2    |
|RD0038681 |2    |
|RD0005163 |2    |
|RD0031994 |2    |
|RD0004237 |2    |
|RD0041995 |2    |
|RD0064414 |2    |
|RD0022672 |2    |
|RD0059712 |2    |
|RD0005260 |2    |
|RD

F. Lazy execution, caching and persistence

In [27]:
lazy_pipeline_df = (
    readings_df.filter((F.col("reading_status") == "Valid") & F.col("energy_kwh").isNotNull())
    .withColumn("energy_charge_estimate", F.col("energy_kwh") * F.lit(1.0))
    .groupBy("meter_id")
    .agg(F.sum("energy_charge_estimate").alias("total_energy_estimate"))
    .orderBy(F.col("total_energy_estimate").desc())
)
lazy_pipeline_df.explain("formatted")
# The plan is inspected above before triggering an action.
lazy_pipeline_df.show(10)


== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan csv  (1)


(1) Scan csv 
Output [3]: [meter_id#18, energy_kwh#20, reading_status#24]
Batched: false
Location: InMemoryFileIndex [file:/content/meter_readings_90000.csv]
PushedFilters: [IsNotNull(reading_status), EqualTo(reading_status,Valid), IsNotNull(energy_kwh)]
ReadSchema: struct<meter_id:string,energy_kwh:double,reading_status:string>

(2) Filter
Input [3]: [meter_id#18, energy_kwh#20, reading_status#24]
Condition : ((isnotnull(reading_status#24) AND (reading_status#24 = Valid)) AND isnotnull(energy_kwh#20))

(3) Project
Output [2]: [meter_id#18, (energy_kwh#20 * 1.0) AS energy_charge_estimate#1171]
Input [3]: [meter_id#18, energy_kwh#20, reading_status#24]

(4) HashAggregate
Input [2]: [meter_id#18, energy_charge_estimate#1171]
Keys [1

In [28]:
valid_readings_cached_df = readings_df.filter(F.col("reading_status") == "Valid").cache()
start = time.perf_counter()
valid_count_1 = valid_readings_cached_df.count()
elapsed_1 = time.perf_counter() - start
start = time.perf_counter()
valid_count_2 = valid_readings_cached_df.count()
elapsed_2 = time.perf_counter() - start
print("Valid count 1:", valid_count_1, "| elapsed seconds:", round(elapsed_1, 4))
print("Valid count 2:", valid_count_2, "| elapsed seconds:", round(elapsed_2, 4))
print("is_cached:", valid_readings_cached_df.is_cached)
valid_readings_cached_df.unpersist()


Valid count 1: 74486 | elapsed seconds: 3.014
Valid count 2: 74486 | elapsed seconds: 0.217
is_cached: True


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [29]:
outage_readings_df = readings_df.filter(F.col("outage_minutes") > 0).persist(StorageLevel.MEMORY_AND_DISK)
print("Outage rows:", outage_readings_df.count())
print("Total outage minutes:", outage_readings_df.agg(F.sum("outage_minutes").alias("total")).first()["total"])
outage_readings_df.unpersist()


Outage rows: 7653
Total outage minutes: 707706


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [31]:
def timed_aggregation(df, label):
    start = time.perf_counter()
    result = df.groupBy("source_type").agg(F.sum("energy_kwh").alias("total_energy")).count()
    elapsed = time.perf_counter() - start
    print(label, "result groups:", result, "elapsed seconds:", round(elapsed, 4))

timed_aggregation(readings_df, "Without caching 1")
timed_aggregation(readings_df, "Without caching 2")
readings_cached_for_benchmark_df = readings_df.cache()
readings_cached_for_benchmark_df.count()  # materialize cache
timed_aggregation(readings_cached_for_benchmark_df, "With caching - run 1")
timed_aggregation(readings_cached_for_benchmark_df, "With caching - run 2")
readings_cached_for_benchmark_df.unpersist()


Without caching 1 result groups: 2 elapsed seconds: 0.6967
Without caching 2 result groups: 2 elapsed seconds: 0.5998
With caching - run 1 result groups: 2 elapsed seconds: 0.5553
With caching - run 2 result groups: 2 elapsed seconds: 0.714


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

G. Partitions and output

In [32]:
readings_8_partitions_df = readings_df.repartition(8)
partition_counts_df = (
    readings_8_partitions_df.withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id").count().orderBy("partition_id")
)
partition_counts_df.show()


+------------+-----+
|partition_id|count|
+------------+-----+
|           0|11249|
|           1|11249|
|           2|11251|
|           3|11251|
|           4|11250|
|           5|11250|
|           6|11250|
|           7|11250|
+------------+-----+



In [33]:
readings_12_partitions_df = readings_df.repartition(12)
coalesced_3_df = readings_12_partitions_df.coalesce(3)
repartitioned_3_df = readings_12_partitions_df.repartition(3)
print("Initial partitions:", readings_12_partitions_df.rdd.getNumPartitions())
print("coalesce partitions:", coalesced_3_df.rdd.getNumPartitions())
print("repartition(3) partitions:", repartitioned_3_df.rdd.getNumPartitions())
coalesced_3_df.explain("formatted")
repartitioned_3_df.explain("formatted")


Initial partitions: 12
coalesce partitions: 3
repartition(3) partitions: 3
== Physical Plan ==
AdaptiveSparkPlan (7)
+- == Final Plan ==
   ResultQueryStage (5)
   +- Coalesce (4)
      +- ShuffleQueryStage (3), Statistics(sizeInBytes=13.0 MiB, rowCount=9.00E+4)
         +- Exchange (2)
            +- Scan csv  (1)
+- == Initial Plan ==
   Coalesce (6)
   +- Exchange (2)
      +- Scan csv  (1)


(1) Scan csv 
Output [11]: [reading_id#17, meter_id#18, reading_timestamp#19, energy_kwh#20, voltage_v#21, power_factor#22, outage_minutes#23, reading_status#24, source_type#25, ambient_temperature_c#26, operator_id#27]
Batched: false
Location: InMemoryFileIndex [file:/content/meter_readings_90000.csv]
ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:double,power_factor:double,outage_minutes:int,reading_status:string,source_type:string,ambient_temperature_c:double,operator_id:string>

(2) Exchange
Input [11]: [reading_id#17, meter_id#1

In [34]:
cleaned_readings_for_output_df = readings_nonnull_df.withColumn(
    "reading_timestamp", F.to_timestamp("reading_timestamp")
)
enriched_readings_df = (
    cleaned_readings_for_output_df
    .join(meter_df, on="meter_id", how="left")
    .join(substation_df, on="substation_id", how="left")
)
parquet_output_path = "/content/enriched_energy_readings_parquet"
(enriched_readings_df.repartition(4)
 .write.mode("overwrite").parquet(parquet_output_path))
print("Parquet part files:")
print([f for f in os.listdir(parquet_output_path) if f.startswith("part-")])


Parquet part files:
['part-00000-1d55dbc8-c68c-4b6d-b6a5-59d5af2e1548-c000.snappy.parquet', 'part-00003-1d55dbc8-c68c-4b6d-b6a5-59d5af2e1548-c000.snappy.parquet', 'part-00002-1d55dbc8-c68c-4b6d-b6a5-59d5af2e1548-c000.snappy.parquet', 'part-00001-1d55dbc8-c68c-4b6d-b6a5-59d5af2e1548-c000.snappy.parquet']


Final Capstone

In [36]:
# 1.
capstone_union_df = readings_df.unionByName(readings_extra_df)
capstone_dedup_df = capstone_union_df.dropDuplicates(["reading_id"])
print("Union rows:", capstone_union_df.count())
print("Deduplicated rows:", capstone_dedup_df.count())

# 2
capstone_clean_df = capstone_dedup_df.filter(F.col("energy_kwh").isNotNull())
capstone_billable_df = capstone_clean_df.filter(F.col("reading_status") != "Rejected")

# 3.
capstone_enriched_df = (
    capstone_billable_df
    .withColumn("reading_timestamp", F.to_timestamp("reading_timestamp"))
    .join(meter_df, on="meter_id", how="left")
    .join(substation_df, on="substation_id", how="left")
    .join(tariff_df, on="tariff_id", how="left")
)

# 4.
capstone_enriched_df = capstone_enriched_df.withColumn(
    "month", F.date_format("reading_timestamp", "yyyy-MM")
).withColumn(
    "energy_charge_inr", F.col("energy_kwh") * F.col("rate_per_kwh_inr")
)

# 5.
capstone_report_df = (
    capstone_enriched_df.groupBy("month", "region", "site_type")
    .agg(
        F.sum("energy_kwh").alias("total_kwh"),
        F.sum("energy_charge_inr").alias("estimated_energy_charges_inr"),
        F.sum(F.coalesce(F.col("outage_minutes"), F.lit(0))).alias("total_outage_minutes"),
        F.count(F.lit(1)).alias("total_readings"),
        F.sum(F.when(F.col("outage_minutes") > 0, 1).otherwise(0)).alias("outage_readings")
    )
    .withColumn(
        "outage_reading_percentage",
        F.when(F.col("total_readings") > 0,
               F.col("outage_readings") * 100.0 / F.col("total_readings"))
         .otherwise(F.lit(0.0))
    )
)

# 6.
capstone_rank_window = Window.partitionBy("month", "region").orderBy(
    F.col("estimated_energy_charges_inr").desc_nulls_last()
)
capstone_report_df = capstone_report_df.withColumn(
    "site_type_charge_rank", F.dense_rank().over(capstone_rank_window)
)

# 7.
capstone_report_cached_df = capstone_report_df.cache()
print("Capstone report row count:", capstone_report_cached_df.count())
capstone_report_cached_df.orderBy("month", "region", "site_type_charge_rank").show(50, truncate=False)
capstone_report_cached_df.unpersist()

# 8.
capstone_output_path = "/content/energy_consumption_reliability_report_parquet"
(capstone_report_df.repartition(2)
 .write.mode("overwrite").parquet(capstone_output_path))
print("Capstone Parquet part files:")
print([f for f in os.listdir(capstone_output_path) if f.startswith("part-")])


Union rows: 102000
Deduplicated rows: 101300
Capstone report row count: 120
+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+---------------+-------------------------+---------------------+
|month  |region |site_type      |total_kwh         |estimated_energy_charges_inr|total_outage_minutes|total_readings|outage_readings|outage_reading_percentage|site_type_charge_rank|
+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+---------------+-------------------------+---------------------+
|2026-01|Central|Industrial     |24871.22          |226328.102                  |4183                |418           |47             |11.24401913875598        |1                    |
|2026-01|Central|Commercial     |10649.029999999995|87322.04599999999           |4876                |567           |55             |9.700176366843033        |2                    |
|2026-01|Centr